# Polygons

## Import resources

In [ ]:
import pyku
import pyku.resources as resources
import pyku.visualisation as visualisation

## Download directory

Polygons are mostly downloaded on-the-fly. The download takes place only one time and the downloaded data are reused if the polygons are called twice. The download path can be shown with. If you set the environment variable ``PYKU_DATA_DIR``, the data will automatically be downloaded in that directory. If you do not set that environment variable, data are downloaded by default in the *standard* directory ``~/.local/share/pyku``. You can also overwrite the data_dir during runtime using ``pyku.PYKU_CONFIG.set(data_dir) = <your_new_data_dir>``, but there wont be any sanity checks that way, so the directory needs to exist and needs to have the right permissions set.

In [ ]:
pyku.PYKU_CONFIG

## List of available polygons

The identifier of all available polygons can be accessed like so:

In [ ]:
resources.list_polygon_identifiers()

## Germany

In [ ]:
geodf = resources.get_geodataframe('germany')

print(f"{resources.pyku_resources.get('polygons').get('germany').get('description')}")

visualisation.regions(
    geodf,
    area_def='HYR-GER-LAEA-5',
    size_inches=(5, 5)
)

## German cities

In [ ]:
# Get German cities
# -----------------

geodf = resources.get_geodataframe('german_cities')

# Get description
# ---------------

print(f"{resources.pyku_resources.get('polygons').get('german_cities').get('description')}")

# Select 'Kreisfreie Stadt'
# -------------------------

geodf = geodf.query("BEZ == 'Kreisfreie Stadt'")

# Select cities
# -------------

geodf = geodf[geodf['GEN'].isin(['Frankfurt', 'Offenbach', 'Berlin', 'Hamburg', 'München'])]

# Plot
# ----

visualisation.regions(
    geodf,
    area_def='HYR-GER-LAEA-5',
    size_inches=(5, 5)
)

## The alps

Load regions with

In [ ]:
geodf = resources.get_geodataframe('ne_10m_geography_regions_polys')

Search any rows that would contain the word alps

In [ ]:
rows = (geodf.map(lambda x: str(x).lower() == 'alps')).any(axis=1)
geodf.loc[rows]

In [ ]:
# Get the Alps
# ------------

geodf = geodf.query("NAME == 'ALPS'")

# Plot
# ----

visualisation.regions(
    geodf,
    area_def='euradcom_ps',
    size_inches=(5, 5)
)

## Natural areas of Germany

In [ ]:
geodf = resources.get_geodataframe('natural_areas_of_germany')

# Plot
# ----

visualisation.regions(
    geodf,
    area_def='HYR-LAEA-5',
    size_inches=(8, 8),
)

The merged polygons can be accessed with:

In [ ]:
geodf = resources.get_geodataframe('natural_areas_of_germany_merged4')

display(geodf)

# Plot
# ----

visualisation.regions(
    geodf,
    area_def='HYR-LAEA-5',
    size_inches=(8, 8),
)

## Climatic Natural Regions of Germany

In [ ]:
geodf = resources.get_geodataframe('natural_climatic_regions_of_germany')

display(geodf)

# Plot
# ----

visualisation.regions(
    geodf,
    area_def='HYR-GER-LAEA-5',
    size_inches=(8, 8),
)

## Germany split into western, northern, easter and south Germany

In [ ]:
geodf = resources.get_geodataframe('german_directions')

# Plot
# ----

visualisation.regions(
    geodf,
    area_def='HYR-GER-LAEA-5',
    size_inches=(8, 8),
)

## German states

In [ ]:
geodf = resources.get_geodataframe('german_states')

visualisation.regions(
    geodf,
    area_def='HYR-GER-LAEA-5',
    size_inches=(8, 8),
)

## World

Polygons for the whole world is quite easy.

In [ ]:
geodf = resources.get_geodataframe('ne_10m_coastline')

visualisation.regions(
    geodf,
    area_def='world_360x180',
    size_inches=(8, 8),
)

## Configuration file

The data are configured per yaml file. Users do not need to worry about it unless new polygons are needed. The source of the data is available in the description. You can get the configuration as a python dictionary like so

In [ ]:
import pyku.resources as resources
resources.pyku_resources

## Bringing grids to polygons

pyku provides separate operations for preserving each region's masked grid,
reducing grid values to one value per polygon, and clipping a grid to a
single polygon. The examples below use packaged resources.

In [ ]:
import pyku
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

ds = pyku.resources.get_test_data('hyras_tas')
gdf = pyku.resources.get_geodataframe(
    'natural_climatic_regions_of_germany'
)

geometry_ds = pyku.resources.get_geometry_dataset(
    'natural_climatic_regions_of_germany'
)

# Grid-preserving masking: typically region, time, y, x.
masked = ds.pyku.regionalize(gdf)
masked.tas.dims

# Reduced aggregation: typically geometry, time.
regional = ds.pyku.aggregate_by_region(
    gdf,
    statistic='mean',
    cell_selection='centroid',
)
regional.tas.dims

# Polygon values remain an xvec-indexed Dataset for geometry-aware plots.
pyku.visualisation.regional_value_map(
    regional.isel(time=0),
    var='tas',
    area_def='HYR-GER-LAEA-1',
    figsize=(4, 6),
    cmap='Blues',
)
plt.show()

In [ ]:
for statistic in ['mean', 'min', 'max', 'sum', 'std']:
    regional = pyku.features.aggregate_by_region(
        ds,
        gdf,
        statistic=statistic,
        cell_selection='centroid',
    )
    pyku.visualisation.regional_value_map(
        regional.isel(time=0),
        var='tas',
        area_def='HYR-GER-LAEA-1',
        figsize=(4, 6),
        cmap='Blues',
    )
    plt.show()

In [ ]:
from pyku.features import clip_to_geometry

# Convert the grid to a polygon tree structure and retrieve the corresponding grid points.
area_def = pyku.geo.load_area_def('HYR-GER-LAEA-5')

test_dataset = ds.pyku.project(area_def.name)
test_polygons = geometry_ds.xvec.to_crs(geometry=area_def.to_cartopy_crs())
test_klinat = test_polygons.geometry.values[4]

fig, axes = plt.subplots(
    1, 4,
    figsize=(20, 4),
    subplot_kw={"projection": area_def.to_cartopy_crs()}
)

for ax, method in zip(axes, ["cover", "centroid", "intersects", "envelope"]):
    test_hull = clip_to_geometry(
        test_dataset,
        test_klinat,
        method=method,
    )

    test_hull.isel(time=0)["tas"].plot(
        ax=ax,
        transform=area_def.to_cartopy_crs(),
        cmap="viridis",
        alpha=0.7,
    )

    polys = test_klinat.geoms if hasattr(test_klinat, "geoms") else [test_klinat]
    for poly in polys:
        x, y = poly.exterior.xy
        ax.plot(x, y, color="red", linewidth=2)

    ax.set_title(f"method={method}")



In [ ]:
from pyku.features import _grid_to_polytree, _gen_weights
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pyku.geo as geo

plot_crs = geo.load_area_def('HYR-GER-LAEA-5').to_cartopy_crs()
test_klinat = geometry_ds.xvec.to_crs(geometry=plot_crs).geometry.values[4]


fig, axes = plt.subplots(
    1, 2,
    figsize=(24, 6),
    subplot_kw={"projection": plot_crs}
)

for ax, how in zip(axes, ["centroid", "area"]):

    weight_pairs = _gen_weights(test_dataset, [test_klinat], how=how)
    iy_idx, ix_idx = np.unravel_index(weight_pairs[0][0], (test_dataset.sizes['y'], test_dataset.sizes['x']))

    weighted_hull = test_hull.isel(time=0)['tas'].copy(deep=True).astype(float)
    weighted_hull[:] = np.nan

    for iy, ix, weight_value in zip(iy_idx, ix_idx, weight_pairs[0][1]):
        y_coord = test_dataset.y.values[iy]
        x_coord = test_dataset.x.values[ix]
        if y_coord in weighted_hull.y.values and x_coord in weighted_hull.x.values:
            weighted_hull.loc[dict(y=y_coord, x=x_coord)] = weight_value

    weighted_hull.plot(
        ax=ax,
        transform=plot_crs,
        cmap='Blues',
        vmin=0,
        add_colorbar=True,
    )

    geometries = (
        test_klinat.geoms
        if hasattr(test_klinat, "geoms")
        else [test_klinat]
    )
    for poly in geometries:
        x, y = poly.exterior.xy
        ax.plot(x, y, color='red', linewidth=2)
        ax.fill(x, y, color='red', alpha=0.2)

    ax.set_title(f'Grid-cell contribution weights method={how}')
